# Introducción a Pandas II: Agrupaciones

In [ ]:
import pandas as pd

In [ ]:
data_imp = pd.read_excel("importaciones_exportaciones.xlsx", sheet_name="Importaciones")

## Top 5 de los paises con mayores importaciones

In [ ]:
# paises unicos en la base de datos
paises = data_imp["País"].unique()

# lista para guardar el total de importaciones
importaciones_totales = []

for pais_i in paises:
    importacion_i = data_imp.loc[ data_imp["País"] == pais_i, "Valor_USD" ].sum()
    importaciones_totales.append((pais_i, importacion_i))

sorted( importaciones_totales, key= lambda x: x[1], reverse=True )

[('Reino Unido', np.float64(65958992.779999994)),
 ('Canadá', np.float64(55994052.809999995)),
 ('India', np.float64(53249225.07)),
 ('Japón', np.float64(51712178.34000001)),
 ('Estados Unidos', np.float64(51502809.31000001)),
 ('México', np.float64(50140780.14)),
 ('Brasil', np.float64(48974174.620000005)),
 ('Alemania', np.float64(47530769.059999995)),
 ('Francia', np.float64(45768254.25)),
 ('China', np.float64(41876504.45999999))]

Agrupaciones con Pandas

In [ ]:
import plotly.express as px

In [ ]:
importaciones_totales = data_imp.groupby("País")["Valor_USD"].sum().nlargest(5).reset_index()

fig = px.bar( importaciones_totales, x = "País", y = "Valor_USD")
fig.show()

importaciones anuales de café


In [ ]:
imp_cafe = data_imp.loc[ data_imp["Producto"] == "Café" ].groupby("Año")["Valor_USD"].sum().reset_index()
px.bar( imp_cafe, x = "Año", y = "Valor_USD", title = "Importaciones anuales de café")

Proporción de exportaciones por producto

In [ ]:
data_exp = pd.read_excel("importaciones_exportaciones.xlsx", sheet_name="Exportaciones")

In [ ]:
exp_productos = data_exp.groupby("Producto")["Valor_USD"].sum().reset_index()
px.pie(exp_productos, values = "Valor_USD", names = "Producto")

Proporción de exportaciones por país

In [ ]:
exp_paises = data_exp.groupby("País")["Valor_USD"].sum().reset_index()
px.pie( exp_paises, values = "Valor_USD", names = "País")

# Limpieza de información y Agregaciones

In [ ]:
data = pd.read_csv("gun-violence-data_01-2013_03-2018.csv")

Selección de información de interés

In [ ]:
data.columns

Index(['incident_id', 'date', 'state', 'city_or_county', 'address', 'n_killed',
       'n_injured', 'incident_url', 'source_url',
       'incident_url_fields_missing', 'congressional_district', 'gun_stolen',
       'gun_type', 'incident_characteristics', 'latitude',
       'location_description', 'longitude', 'n_guns_involved', 'notes',
       'participant_age', 'participant_age_group', 'participant_gender',
       'participant_name', 'participant_relationship', 'participant_status',
       'participant_type', 'sources', 'state_house_district',
       'state_senate_district'],
      dtype='object')

Formato de columnas de tiempo

In [ ]:
from datetime import datetime

columnas = [
    "date",
    "state",
    "n_killed",
    "n_injured",
    "gun_type",
    "participant_age",
    "participant_gender"
]
d = data[columnas]

# transformar a formato de fecha
date = pd.to_datetime(d["date"])

# insertar nuevas columnas
d.insert(1, "year", date.dt.year)
d.insert(2, "month", date.dt.month)
d.insert(3, "day", date.dt.day)
d.insert(4, "day_of_week", date.dt.day_of_week)
d.insert(5, "n_victims", d["n_killed"] + d["n_injured"])

In [ ]:
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239677 entries, 0 to 239676
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   date                239677 non-null  object
 1   year                239677 non-null  int32 
 2   month               239677 non-null  int32 
 3   day                 239677 non-null  int32 
 4   day_of_week         239677 non-null  int32 
 5   n_victims           239677 non-null  int64 
 6   state               239677 non-null  object
 7   n_killed            239677 non-null  int64 
 8   n_injured           239677 non-null  int64 
 9   gun_type            140226 non-null  object
 10  participant_age     147379 non-null  object
 11  participant_gender  203315 non-null  object
dtypes: int32(4), int64(3), object(5)
memory usage: 18.3+ MB


In [ ]:
m  = d["day_of_week"].map({
    0: "Monday", 1: "Tuesday",
    2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday"
})
d.insert(6, "day_", m)

Numero de victimas por estado

In [ ]:
victimas_per_state = d.groupby("state")["n_victims"].sum().sort_values().reset_index()
px.bar( victimas_per_state, x = "state", y = "n_victims")

In [ ]:
victimas_por_año = d.groupby("month")["n_victims"].sum().sort_values().reset_index()
px.bar( victimas_por_año, x = "month", y = "n_victims")

In [ ]:
victimas_por_dia = d.groupby("day_")["n_victims"].sum().sort_values().reset_index()
px.bar( victimas_por_dia, x = "day_", y = "n_victims")

Numero de victimas para Illinois en función del mes

In [ ]:
victimas_illionois_mes = d.loc[d["state"] == "Illinois"].groupby("month")["n_victims"].sum().reset_index()
px.bar( victimas_illionois_mes, x = "month", y = "n_victims" )

Numero de victimas para Illinois en función del año

In [ ]:
victimas_illionois_año = d.loc[d["state"] == "Illinois"].groupby("year")["n_victims"].sum().reset_index()
px.line( victimas_illionois_año, x = "year", y = "n_victims" )